<a href="https://colab.research.google.com/github/scudilio/AI---Deep-Learning/blob/main/Aula%202/MBA_DL_Aula02_02_colab_deteccao_objetos_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 2 | Aplicação 2 (Colab): detectar e contar objetos em fotos e vídeo

**MBA FIAP | Deep Learning e Redes Neurais Modernas | Profa. Juliana Scudilio**

**Problema de negócio.** Contar pessoas numa loja, produtos numa gôndola, veículos num pátio. Não basta saber *o que* há na imagem: é preciso saber *onde* e *quantos*. Isso é detecção de objetos.

**O que muda em relação à aplicação 1.** Lá treinamos uma rede do zero com 9.000 imagens e chegamos a 60%. Aqui **não treinamos nada**: usamos um detector já treinado em mais de 100 mil fotos (COCO, 80 classes do cotidiano).

**Modelo.** YOLOv8n ("nano"): a imagem passa uma única vez pela rede, que devolve caixas, classes e confiança.

**Roteiro.** 1. Instalar → 2. Carregar o modelo → 3. Fotos de exemplo → 4. Limiar de confiança → 5. Contagem → 6. Sua própria foto → 7. Vídeo quadro a quadro → 8. Custo e discussão.

Este notebook roda no Google Colab. Não precisa de GPU; com GPU (Ambiente de execução → Alterar tipo → GPU) o vídeo fica mais rápido.

## 1. Instalar a biblioteca

A `ultralytics` carrega o YOLO. O Colab já tem PyTorch, OpenCV e scikit-image. Leva cerca de 1 minuto.

In [ ]:
!pip install --quiet ultralytics imageio imageio-ffmpeg

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from skimage import data as exemplos        # fotos de exemplo embutidas no scikit-image (domínio público / CC0)

print("torch", torch.__version__, "| GPU?", torch.cuda.is_available())

⏱ **Checkpoint 1:** a célula acima imprimiu a versão sem erro?

## 2. Carregar o detector pré-treinado

`yolov8n.pt` guarda a arquitetura e os pesos aprendidos no dataset COCO. Se o arquivo não estiver na pasta, a biblioteca baixa (6 MB). Nada será treinado aqui.

In [ ]:
modelo = YOLO("yolov8n.pt")                 # baixa os pesos na primeira vez
classes = modelo.names                      # dicionário {id: nome}, 80 classes do COCO
print(len(classes), "classes, por exemplo:", [classes[i] for i in [0, 1, 2, 5, 15, 16, 41, 56, 62, 67]])

## 3. Detectar em fotos de exemplo

Três fotos que vêm junto com o scikit-image: uma astronauta (pessoa), uma mesa com café e uma gata.

Para cada foto, o modelo devolve uma lista de caixas. Cada caixa tem: coordenadas, classe e **confiança** (0 a 1).

In [ ]:
def detectar(img_rgb, conf=0.25):
    """Roda o detector numa imagem RGB e devolve (resultado, imagem anotada em RGB)."""
    res = modelo.predict(img_rgb[..., ::-1], conf=conf, verbose=False)[0]   # a biblioteca espera BGR (convenção do OpenCV)
    return res, res.plot()[..., ::-1]

def listar(res):
    return [(classes[int(c)], round(float(p), 2)) for c, p in zip(res.boxes.cls, res.boxes.conf)]

fotos = {"astronauta": exemplos.astronaut(), "cafe": exemplos.coffee(), "gata": exemplos.chelsea()}

_ = detectar(fotos["gata"])   # primeira chamada é mais lenta (prepara o modelo); ignoramos no tempo

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (nome, img) in zip(axes, fotos.items()):
    t0 = time.time(); res, anotada = detectar(img); dt = time.time() - t0
    ax.imshow(anotada); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{nome}: {listar(res)}\n{dt*1000:.0f} ms", fontsize=9)
plt.tight_layout(); plt.show()

Cada caixa traz o nome da classe e a confiança. A "mesa" (dining table) aparece com confiança mais baixa que a "xícara" (cup): o modelo está menos seguro.

⏱ **Checkpoint 2:** as três fotos apareceram com caixas?

## 4. O limiar de confiança é uma decisão sua

O modelo gera dezenas de caixas candidatas. O **limiar** (`conf`) decide a partir de que confiança uma caixa "existe". Mesma foto, três limiares.

In [ ]:
img = fotos["cafe"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, limiar in zip(axes, [0.05, 0.25, 0.60]):
    res, anotada = detectar(img, conf=limiar)
    ax.imshow(anotada); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"limiar {limiar}: {len(res.boxes)} objetos  {listar(res)}", fontsize=9)
plt.tight_layout(); plt.show()

**Leitura de negócio.** Limiar baixo encontra mais e inventa mais. Limiar alto só mostra o que tem certeza e perde objetos difíceis.
Não existe limiar certo: existe o limiar certo *para o custo do erro* naquele processo.

## 5. Contar objetos por classe

Detecção vira dado de negócio quando vira contagem.

In [ ]:
from collections import Counter

def contar(res):
    return Counter(classes[int(c)] for c in res.boxes.cls)

for nome, img in fotos.items():
    res, _ = detectar(img, conf=0.25)
    print(f"{nome:<10} ->", dict(contar(res)))

## 6. A sua própria foto (opcional)

Rode a célula e escolha uma foto do seu computador (rua, escritório, mesa com objetos). Se cancelar, a célula é pulada.

In [ ]:
img_propria = None
try:
    from google.colab import files
    enviados = files.upload()                        # abre o seletor de arquivos do Colab
    if enviados:
        from PIL import Image
        nome = list(enviados)[0]
        img_propria = np.asarray(Image.open(nome).convert("RGB"))
except ImportError:
    print("Fora do Colab: pule esta célula.")

if img_propria is not None:
    res, anotada = detectar(img_propria, conf=0.25)
    plt.figure(figsize=(10, 7)); plt.imshow(anotada); plt.axis("off"); plt.title(str(dict(contar(res)))); plt.show()

## 7. Vídeo: o mesmo detector, quadro a quadro

Um vídeo é uma sequência de imagens. Detectar em vídeo é rodar o detector em cada quadro e acumular: contagem ao longo do tempo, alertas.

Usamos o vídeo de exemplo da disciplina, `video_exemplo.mp4` (90 quadros, montado a partir das fotos acima com movimento de câmera). A célula procura o arquivo na pasta; se não achar, pede para você enviar.

In [ ]:
import imageio

candidatos = ["video_exemplo.mp4", "../data/video_exemplo.mp4", "data/video_exemplo.mp4"]
caminho_video = next((c for c in candidatos if os.path.exists(c)), None)
if caminho_video is None:
    from google.colab import files
    print("Envie o arquivo video_exemplo.mp4 (ou um vídeo curto seu):")
    caminho_video = list(files.upload())[0]

leitor = imageio.get_reader(caminho_video, "ffmpeg")
quadros = np.stack([q for q in leitor]); leitor.close()   # (n_quadros, altura, largura, 3), RGB
print("Quadros:", quadros.shape[0], "| resolução:", quadros.shape[2], "x", quadros.shape[1])

In [ ]:
PASSO = 3                                                      # analisa 1 a cada 3 quadros para ser rápido
anotados, contagens, tempos = [], [], []
for i in range(0, len(quadros), PASSO):
    t0 = time.time(); res, anotada = detectar(quadros[i], conf=0.3); tempos.append(time.time() - t0)
    anotados.append(anotada); contagens.append(contar(res))

# 8 quadros espalhados pelo vídeo
fig, axes = plt.subplots(2, 4, figsize=(16, 6.5))
for ax, k in zip(axes.ravel(), np.linspace(0, len(anotados) - 1, 8).astype(int)):
    ax.imshow(anotados[k]); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"quadro {k*PASSO}: {dict(contagens[k])}", fontsize=9)
plt.suptitle("Detecção quadro a quadro"); plt.tight_layout(); plt.show()

print(f"Tempo médio por quadro: {np.mean(tempos)*1000:.0f} ms  ->  {1/np.mean(tempos):.1f} quadros por segundo")

In [ ]:
# Contagem ao longo do tempo: o dado que alimentaria um painel de negócio
todas = sorted({c for ct in contagens for c in ct})
plt.figure(figsize=(10, 3.5))
for c in todas:
    plt.plot([ct.get(c, 0) for ct in contagens], label=c, marker="o", ms=3)
plt.xlabel(f"quadro analisado (1 a cada {PASSO})"); plt.ylabel("objetos detectados"); plt.legend(); plt.grid(alpha=.3)
plt.title("Quantos objetos de cada classe em cada quadro"); plt.show()

Repare se algum quadro conta a mesma pessoa duas vezes (duas caixas sobrepostas). É um erro clássico de detector em movimento; subir o limiar costuma eliminar a caixa mais fraca.

### Ver o vídeo com as caixas

A célula grava um vídeo anotado e mostra aqui mesmo no notebook.

In [ ]:
from IPython.display import HTML
import base64

saida = "video_detectado.mp4"
escritor = imageio.get_writer(saida, fps=8, codec="libx264", macro_block_size=1)
for a in anotados:
    escritor.append_data(a)
escritor.close()

video_b64 = base64.b64encode(open(saida, "rb").read()).decode()
HTML(f'<video width="480" controls autoplay loop muted src="data:video/mp4;base64,{video_b64}"></video>')

⏱ **Checkpoint 3:** quantos quadros por segundo a sua máquina processou?

## 8. Latência e custo: por que isso importa para o negócio

- Um vídeo comum tem 30 quadros por segundo. Se a máquina processa menos que isso, "tempo real" exige GPU, hardware dedicado ou analisar só parte dos quadros (como fizemos com `PASSO = 3`).
- O modelo "nano" é o menor da família. Versões maiores acertam mais e custam mais por quadro. É uma escolha de engenharia e de orçamento.
- Nada aqui foi treinado. Se o seu problema tem classes que o COCO não conhece (uma peça específica, um defeito), é preciso **adaptar** o modelo com os seus dados. O notebook 04 mostra a ideia (transfer learning) e a aula 4 aprofunda.

## Para discutir

1. Que contagem, na sua empresa, hoje é feita por uma pessoa olhando e poderia vir de uma câmera?
2. Qual limiar você escolheria para esse caso, e por quê?
3. O que acontece com esse modelo se a câmera for trocada ou a iluminação mudar? Quem percebe?